In [0]:
import pyspark.sql.functions as F

In [0]:
# ============================================================
# 1. SOURCE CONFIGURATION
# ============================================================

# Path of the raw NYC Taxi source file in our Unity Catalog Volume.
source_path = (
    "/Volumes/workspace/"
    "nyc_taxi_bronze/raw_volume/"
    "yellow_tripdata_2025-01.parquet"
)


# ============================================================
# 2. READ SOURCE DATA
# ============================================================

# Read the Parquet file using Spark.
df = spark.read.parquet(source_path)


# ============================================================
# 3. BASIC SOURCE INFORMATION
# ============================================================

# Display the number of records received from the source.
print("Record count:", df.count())

# Display the complete source schema.
df.printSchema()


# ============================================================
# 4. DISPLAY SAMPLE RECORDS
# ============================================================

# Display a small sample so we can visually inspect the data.
display(df.limit(10))

In [0]:
# ============================================================
# 5. NULL PROFILE
# ============================================================

# Get the total number of records in the source dataset.
total_records = df.count()
display(total_records)
# Create a list containing the number of NULL values
# for every column in the source dataset.
null_profile = df.select([
    # For each column:
    # - when the value is NULL, count it as 1
    # - otherwise count it as 0
    #
    # sum() then gives us the total NULL count for that column.
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in df.columns
])

# Display the NULL profile.
display(null_profile)

Check the candidate business key


In [0]:
# ============================================================
# 6. BUSINESS KEY VALIDATION
# ============================================================

# Define the candidate columns that we want to test
# as a possible logical business key.
candidate_key = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
]

# Count all records in the source dataset.
total_key_records = df.count()

# Count distinct combinations of the candidate key columns.
distinct_key_records = (
    df
    .select(*candidate_key)
    .distinct()
    .count()
)

# Calculate how many records are duplicates
# according to the candidate key.
duplicate_key_records = (
    total_key_records - distinct_key_records
)

# Display the results.
print("Total records:", total_key_records)
print("Distinct candidate keys:", distinct_key_records)
print("Duplicate candidate-key records:", duplicate_key_records)

Find actual duplicates

In [0]:
# ============================================================
# 7. SAMPLE DUPLICATE KEYS
# ============================================================

# Group records by the candidate key.
# Any group with count > 1 represents a duplicate key.
duplicate_keys_df = (
    df
    .groupBy(*candidate_key)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

# Display the top duplicate keys.
display(duplicate_keys_df.limit(100))

Determine the data date range

In [0]:
# ============================================================
# 8. DATA DATE RANGE
# ============================================================

# Find the earliest and latest pickup timestamps.
date_range_df = df.select(
    F.min("tpep_pickup_datetime").alias("min_pickup_datetime"),
    F.max("tpep_pickup_datetime").alias("max_pickup_datetime"),
    F.min("tpep_dropoff_datetime").alias("min_dropoff_datetime"),
    F.max("tpep_dropoff_datetime").alias("max_dropoff_datetime")
)

# Display the date range.
display(date_range_df)

Check basic numeric ranges

In [0]:
# ============================================================
# 9. BASIC NUMERIC PROFILE
# ============================================================

# Calculate minimum, maximum and average values
# for important numeric columns.
numeric_profile_df = df.select(
    F.min("trip_distance").alias("min_trip_distance"),
    F.max("trip_distance").alias("max_trip_distance"),
    F.avg("trip_distance").alias("avg_trip_distance"),

    F.min("fare_amount").alias("min_fare_amount"),
    F.max("fare_amount").alias("max_fare_amount"),
    F.avg("fare_amount").alias("avg_fare_amount"),

    F.min("total_amount").alias("min_total_amount"),
    F.max("total_amount").alias("max_total_amount"),
    F.avg("total_amount").alias("avg_total_amount"),

    F.min("passenger_count").alias("min_passenger_count"),
    F.max("passenger_count").alias("max_passenger_count"),
    F.avg("passenger_count").alias("avg_passenger_count")
)

# Display the profile.
display(numeric_profile_df)

Date range

In [0]:
# ============================================================
# DATA DATE RANGE
# ============================================================

# Find the earliest and latest pickup/dropoff timestamps.
date_range_df = df.select(
    F.min("tpep_pickup_datetime").alias("min_pickup_datetime"),
    F.max("tpep_pickup_datetime").alias("max_pickup_datetime"),
    F.min("tpep_dropoff_datetime").alias("min_dropoff_datetime"),
    F.max("tpep_dropoff_datetime").alias("max_dropoff_datetime")
)

# Display the date range.
display(date_range_df)

Determine the real ingestion key and incremental strategy

In [0]:
# ============================================================
# 10. FULL-ROW DUPLICATE ANALYSIS
# ============================================================

# Count the total number of rows in the source dataset.
total_records = df.count()

# Count the number of completely distinct rows across ALL columns.
distinct_full_rows = df.distinct().count()

# Calculate rows that are exact duplicates across every column.
exact_duplicate_records = total_records - distinct_full_rows

# Display the results.
print("Total records:", total_records)
print("Distinct full rows:", distinct_full_rows)
print("Exact duplicate records:", exact_duplicate_records)

One final investigation

In [0]:
# ============================================================
# 13. FIND A REAL DUPLICATE CANDIDATE KEY
# ============================================================

# Find candidate-key combinations that occur more than once.
duplicate_key_sample = (
    df
    .groupBy(
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime"
    )
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .limit(1)
)

# Display one candidate-key combination that occurs multiple times.
display(duplicate_key_sample)

In [0]:
# ============================================================
# 15. INSPECT THE FOUR RECORDS
# ============================================================

# Filter the source DataFrame using the candidate key
# that we found to have four occurrences.
duplicate_records_df = (
    df
    .filter(
        (F.col("VendorID") == 2)
        & (
            F.col("tpep_pickup_datetime")
            == "2025-01-11T00:08:00.000"
        )
        & (
            F.col("tpep_dropoff_datetime")
            == "2025-01-11T00:19:00.000"
        )
    )
)

# Display all four records.
display(duplicate_records_df)

In [0]:
# ============================================================
# 16. SOURCE COLUMN INVENTORY
# ============================================================

# Loop through every field in the DataFrame schema.
# This allows us to inspect the actual source column names
# and their Spark data types instead of making assumptions.
for field in df.schema.fields:

    # Print the column name and its Spark data type.
    # Example:
    # VendorID: long
    # tpep_pickup_datetime: timestamp
    print(
        f"{field.name}: {field.dataType.simpleString()}"
    )

Build a deterministic record_id

In [0]:
# ============================================================
# 13. GENERATE DETERMINISTIC RECORD ID
# ============================================================

# These are the source columns that represent the actual
# contents of a taxi-trip record.
#
# We explicitly define the columns instead of hashing every
# DataFrame column blindly.
record_hash_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee"
]


# Create a deterministic SHA-256 hash from the selected
# source columns.
#
# concat_ws() combines the values into one string.
# coalesce() converts NULL values to a fixed representation
# so that NULL handling is deterministic.
#
# SHA-256 then creates a fixed-length hexadecimal identifier.
df_with_record_id = (
    df
    .withColumn(
        "record_id",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(
                        F.col(column_name).cast("string"),
                        F.lit("<NULL>")
                    )
                    for column_name in record_hash_columns
                ]
            ),
            256
        )
    )
)


# Display a few records so we can verify the new
# technical identifier.
display(
    df_with_record_id.select(
        "record_id",
        *record_hash_columns
    ).limit(10)
)

Check whether the generated ID is unique

In [0]:
# ============================================================
# 14. VALIDATE RECORD ID UNIQUENESS
# ============================================================

# Count the total number of records.
total_records = df_with_record_id.count()

# Count unique record IDs.
distinct_record_ids = (
    df_with_record_id
    .select("record_id")
    .distinct()
    .count()
)

# Calculate duplicate record IDs.
duplicate_record_ids = (
    total_records - distinct_record_ids
)


# Display the validation results.
print("Total records:", total_records)
print("Distinct record IDs:", distinct_record_ids)
print("Duplicate record IDs:", duplicate_record_ids)

In [0]:
# ============================================================
# CELL 1 — ADD SOURCE FILE PATTERN
# ============================================================

# Add a column that tells the metadata-driven framework
# which source files belong to this pipeline.
spark.sql("""
ALTER TABLE workspace.nyc_taxi_audit.pipeline_metadata
ADD COLUMNS (
    source_file_pattern STRING
)
""")

# Confirm that the column was added successfully.
print("source_file_pattern column added successfully.")

Add record_hash_columns

In [0]:
# ============================================================
# CELL 2 — ADD RECORD HASH COLUMNS
# ============================================================

# Add a column that stores the list of source columns
# used by the framework to generate the deterministic record_id.
spark.sql("""
ALTER TABLE workspace.nyc_taxi_audit.pipeline_metadata
ADD COLUMNS (
    record_hash_columns STRING
)
""")

# Confirm that the column was added successfully.
print("record_hash_columns column added successfully.")

Add incremental_strategy

In [0]:
# ============================================================
# CELL 3 — ADD INCREMENTAL STRATEGY
# ============================================================

# Add a column that tells the metadata-driven framework
# how new data should be identified and processed.
#
# For our NYC Taxi source, we will use FILE-based
# incremental processing because TLC publishes
# the data as monthly files.
spark.sql("""
ALTER TABLE workspace.nyc_taxi_audit.pipeline_metadata
ADD COLUMNS (
    incremental_strategy STRING
)
""")

# Confirm that the column was added successfully.
print("incremental_strategy column added successfully.")

Verify metadata schema

In [0]:
# ============================================================
# CELL 4 — VERIFY METADATA TABLE SCHEMA
# ============================================================

# Read the existing pipeline metadata table from Unity Catalog.
metadata_df = spark.table(
    "workspace.nyc_taxi_audit.pipeline_metadata"
)

# Print the complete schema so we can confirm that
# the three new control columns were added successfully.
metadata_df.printSchema()

Populate the new metadata columns

In [0]:
# ============================================================
# CELL 5 — UPDATE NYC TAXI PIPELINE METADATA
# ============================================================

# Update the existing NYC Yellow Taxi pipeline configuration.
# The framework will use these values later to determine
# how files are discovered, how record IDs are generated,
# and how incremental processing is performed.

spark.sql("""
UPDATE workspace.nyc_taxi_audit.pipeline_metadata

SET

    -- Identify the monthly Yellow Taxi source files.
    source_file_pattern = 'yellow_tripdata_*.parquet',

    -- Store the exact source columns used to generate
    -- the deterministic SHA-256 record_id.
    record_hash_columns =
        'VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee',

    -- Tell the generic framework that the source is processed
    -- incrementally at the FILE level.
    incremental_strategy = 'FILE',

    -- We do not have a source-defined unique business key.
    -- record_id is a technical identifier generated by us,
    -- so we leave business_key as NULL.
    business_key = NULL,

    -- File-level incrementality means we do not need a
    -- timestamp watermark for identifying new source files.
    watermark_column = NULL,

    -- Explicitly identify the ingestion mode.
    load_type = 'FILE_INCREMENTAL',

    -- Record when the metadata was changed.
    updated_at = current_timestamp()

WHERE pipeline_id = 'NYC_YELLOW_001'
""")

# Confirm that the metadata update completed.
print("NYC Taxi pipeline metadata updated successfully.")

Verify the updated metadata

In [0]:
# ============================================================
# CELL 6 — VERIFY UPDATED PIPELINE METADATA
# ============================================================

# Read the NYC Taxi pipeline configuration from the audit table.
metadata_df = (
    spark.table(
        "workspace.nyc_taxi_audit.pipeline_metadata"
    )
    .filter(
        F.col("pipeline_id") == "NYC_YELLOW_001"
    )
)

# Display the configuration so we can verify
# that all ingestion-control values were updated correctly.
display(metadata_df)